# FMI CloudCast → GK2A 파인튜닝 — v3 전 계절판

**목적**: 전 계절 운량 나우캐스트 (겨울 게이트는 v2로 통과 — 1월 +10~17%).

**준비물** (Google Drive `MyDrive/nwp_dl/dataset/`에 npz — 기존에 신규분 추가 업로드):
- `fmi_cloudcast_unet.tar.gz` (사전학습 가중치, 340MB — 기존 그대로)
- `dataset/*.npz` (신규: 11월, 3월, 5월, 6/1~19, 8월 + 기존 12·1월)

**학습/검증 분리 (붙박이 원칙)**: 검증 달(2025-10, 2026-01, 2026-04, 2026-07, 장마 6/20~7/19)은
**학습에 절대 미사용** — 학습은 인접 달만: 2025-11, 2025-12, 2026-03, 2026-05, 2026-06-01~19,
2026-08-01~18. 모니터링 검증은 2026-08-19~21(비게이트 기간).

**모델 규격** (실측): 입력 6채널 = [hist4(10분) | k/12 평면 | 태양고도(도, min-max)], 운량/100, bc+l1.
런타임 → GPU(T4) 선택 후 전체 실행. 라이선스 미명시 자료 — 개인 연구 용도만.

In [ ]:
# Keras 3는 구형 SavedModel 미지원 → tf_keras(Keras 2 호환)로 로드/학습
!pip install -q tf_keras
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/nwp_dl'
import os, glob, tarfile, math, datetime as dt
import numpy as np
import tensorflow as tf
import tf_keras
print('TF', tf.__version__, '| tf_keras', tf_keras.__version__, '| GPU:', tf.config.list_physical_devices("GPU"))

In [ ]:
# 1) 가중치 압축 해제 + tf_keras 로드
MODEL_DIR = '/content/fmi_model'
if not glob.glob(f'{MODEL_DIR}/**/saved_model.pb', recursive=True):
    os.makedirs(MODEL_DIR, exist_ok=True)
    with tarfile.open(f'{BASE}/fmi_cloudcast_unet.tar.gz') as t:
        t.extractall(MODEL_DIR)
cands = [r for r, d, f in os.walk(MODEL_DIR) if 'saved_model.pb' in f]
print('SavedModel:', cands)
model = tf_keras.models.load_model(cands[0], compile=False)
N_CH = int(model.inputs[0].shape[-1])
print('입력:', model.inputs[0].shape)
assert N_CH == 6, f'예상(6채널=[hist4|k/12|sun])과 다름: {N_CH} — 중단, 채널 구성 재확인 필요'

In [ ]:
# 2) 데이터 로드 — Drive 직접 읽기는 끊김(Errno 103) → 로컬 디스크 스테이징(재시도)
#    + RAM 절약: 파일별 배열 리스트 유지(concatenate 시 2배 피크 방지)
import shutil, time
LOCAL = '/content/dataset'
os.makedirs(LOCAL, exist_ok=True)
src = sorted(glob.glob(f'{BASE}/dataset/*.npz'))
print('Drive npz:', len(src))
for f in src:
    dst = os.path.join(LOCAL, os.path.basename(f))
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(f):
        continue
    for a in range(5):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('  재시도', os.path.basename(f), a + 1, e)
            time.sleep(3 * (a + 1))
    else:
        raise RuntimeError(f'복사 실패: {f}')
print('스테이징 완료:', len(glob.glob(f'{LOCAL}/*.npz')))

TRAIN_RANGES = [('20251101','20251130'), ('20251201','20251231'),
                ('20260301','20260331'), ('20260501','20260531'),
                ('20260601','20260619'), ('20260801','20260818')]
VAL_RANGES   = [('20260819','20260821')]
GATE = [('20251001','20251031'), ('20260101','20260131'),
        ('20260401','20260430'), ('20260620','20260719')]  # 로드 자체를 거부

def in_ranges(day, ranges):
    return any(a <= day <= b for a, b in ranges)

def load_ranges(ranges):
    chunks, idx = [], {}
    for f in sorted(glob.glob(f'{LOCAL}/*.npz')):
        day = os.path.basename(f)[:8]
        if not in_ranges(day, ranges):
            continue
        assert not in_ranges(day, GATE), f'게이트 기간 {day}이 학습/검증에 포함됨'
        z = np.load(f)
        fi = len(chunks)
        chunks.append(z['frames'])
        for ri, s in enumerate(z['stamps']):
            idx[str(s)] = (fi, ri)
    print(' 프레임', sum(len(c) for c in chunks))
    return chunks, idx

fr_tr, IDX_TR = load_ranges(TRAIN_RANGES)
fr_va, IDX_VA = load_ranges(VAL_RANGES)

In [ ]:
# 3) 태양고도 채널 — FMI 방식 실측: 고도각(도) 계산 후 프레임별 min-max 0..1
#    (preprocess.py create_sun_elevation_angle → preprocess_single normalize=true)
LATS = np.linspace(46.0, 29.7, 512)[:, None] * np.ones((1, 512))
LONS = np.ones((512, 1)) * np.linspace(113.0, 139.5, 512)[None, :]

def sun_channel(stamp):
    t = dt.datetime.strptime(stamp, '%Y%m%d%H%M')
    doy = t.timetuple().tm_yday
    decl = -23.44 * math.cos(math.radians(360/365*(doy+10)))
    hour = t.hour + t.minute/60
    ha = (hour*15 - 180) + LONS  # 시간각(deg), UTC
    sin_el = (np.sin(np.radians(LATS))*math.sin(math.radians(decl)) +
              np.cos(np.radians(LATS))*math.cos(math.radians(decl))*np.cos(np.radians(ha)))
    el = np.degrees(np.arcsin(np.clip(sin_el, -1, 1)))
    if el.max() > el.min():
        el = (el - el.min()) / np.ptp(el)
    return el.astype(np.float32)

In [ ]:
# 4) 샘플 생성기 — X=[hist4 | k/12 평면 | sun(target)] , y=target(+10*(k+1)분)
#    frames는 파일별 배열 리스트, idx: stamp → (파일번호, 행번호)
STEP, N_HIST, N_LC = 10, 4, 12

def make_sample(frames, idx, s0, k):
    t0 = dt.datetime.strptime(s0, '%Y%m%d%H%M')
    need = [(t0 - dt.timedelta(minutes=STEP*i)).strftime('%Y%m%d%H%M') for i in range(N_HIST-1, -1, -1)]
    need.append((t0 + dt.timedelta(minutes=STEP*(k+1))).strftime('%Y%m%d%H%M'))
    if not all(n in idx for n in need):
        return None
    arrs = [frames[idx[n][0]][idx[n][1]] for n in need]
    if any((a == 255).mean() > 0.1 for a in arrs):
        return None
    hist = [np.where(a == 255, 50, a).astype(np.float32)/100.0 for a in arrs[:N_HIST]]
    y = np.where(arrs[-1] == 255, 50, arrs[-1]).astype(np.float32)/100.0
    lt = np.full((512, 512), k / N_LC, np.float32)
    X = np.stack(hist + [lt, sun_channel(need[-1])], -1)
    return X, y[..., None]

def gen(frames, idx, shuffle=True):
    keys = list(idx.keys())
    while True:
        order = np.random.permutation(len(keys)) if shuffle else range(len(keys))
        for i in order:
            s = make_sample(frames, idx, keys[i], int(np.random.randint(N_LC)))
            if s is not None:
                yield s

sig = (tf.TensorSpec((512, 512, 6), tf.float32), tf.TensorSpec((512, 512, 1), tf.float32))
ds_tr = tf.data.Dataset.from_generator(lambda: gen(fr_tr, IDX_TR), output_signature=sig).batch(4).prefetch(2)
ds_va = tf.data.Dataset.from_generator(lambda: gen(fr_va, IDX_VA, shuffle=False), output_signature=sig).batch(4).take(200)

In [ ]:
# 5) 손실(bc+l1, FMI bcl1 방식) + 파인튜닝  (전 계절: T4 대략 4~6시간)
def bcl1(y_true, y_pred):
    bc = tf_keras.losses.binary_crossentropy(y_true, y_pred)
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred), axis=-1)
    return bc + l1

model.compile(optimizer=tf_keras.optimizers.Adam(1e-5), loss=bcl1, metrics=['mae'])
hist = model.fit(ds_tr, steps_per_epoch=2000, epochs=3, validation_data=ds_va)
# 세션이 끊기면: 런타임 재시작 후 위 셀들 재실행 → epochs를 1~2로 줄여 재개

In [ ]:
# 6) 저장 — 전 계절판은 별도 이름 (겨울 게이트 모델 보존)
out = f'{BASE}/gk2a_allseason.h5'
model.save(out)
print('완료:', out, round(os.path.getsize(out)/2**20), 'MB')
print('→ PC의 kpx-model-charts/dl/ 에 gk2a_allseason.h5 내려받으세요')

In [ ]:
# 7) (선택) 검증기간 사례 눈검증 — +60분(k=5) 예측 vs 실제
import matplotlib.pyplot as plt
keys = sorted(IDX_VA.keys())
smp = make_sample(fr_va, IDX_VA, keys[len(keys)//2], 5)
if smp:
    X, y = smp
    p = model.predict(X[None])[0, ..., 0]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    for a, img, t in zip(ax, [X[..., 3], p, y[..., 0]], ['input t', 'pred +60min', 'actual']):
        a.imshow(img, cmap='gray', vmin=0, vmax=1); a.set_title(t); a.axis('off')
    plt.show()